# A3 — Koopman Feature-Space Geometry

**Goal:** test whether the fixed (untrained, `rff_trainable: false`) dynamics-embedding lift
shows a mechanistic signature — poor local linearizability (eDMD residual) and/or high
sensitivity amplification (Jacobian norm) — on chaotic ODE classes (Lorenz, Rossler, SprottB)
relative to the non-chaotic aperiodic class (Burgers, nu=1.0) and the periodic reference class
(Harmonic), that would explain the A1 behavioral pattern (ablation beats baseline on chaotic
in-distribution/held-out systems; baseline beats ablation on Burgers; ablation beats baseline
on Harmonic).

**This notebook only uses the `baseline_100k` checkpoint** (`use_dynamics_embedding=True`) —
the ablation checkpoint has no lift to analyze.

**Pre-registered interpretation map** (fill in before reading results, don't retrofit):

| Pre-projection (raw dictionary Φ) | Post-projection (392→512 learned) | Reading |
|---|---|---|
| bad on chaotic, good on Burgers | bad on chaotic, good on Burgers | Clean mechanistic confirmation of A1's behavioral split |
| bad on chaotic, good on Burgers | good everywhere | Lift is bad in principle but the trained downstream projection compensates — A1's split needs a *different* explanation (points toward A2a, temporal attention) |
| good everywhere | good everywhere | Negative result — lift geometry does not explain A1 at all → push to A2a |
| bad everywhere (no class separation) | — | Lift is uniformly poor; A1's *selectivity* is not explained by linear-fit quality alone |

**Design decisions locked in (see chat discussion):**
- Both pre-projection (Φ_pre, raw eDMD dictionary: raw patch + random polynomial feats + random
  Fourier feats, concatenated) and post-projection (Φ_post, the learned 392→512 linear map's
  output — what the transformer actually consumes) are tested, kept in **separate result
  tables**, never averaged.
- eDMD is fit **patch-to-patch** (Φ(P_t) → Φ(P_{t+1})), matching the granularity Panda's lift
  actually operates at — NOT the continuous-time Koopman operator.
- **Shared K** (one ridge-regularized linear operator fit across a class-balanced pool of
  trajectories), not per-class K — because the lift is architecturally fixed/global, so a
  per-class-optimal K would test something the model never has access to.
- Ridge regularization, λ chosen by cross-validation (not hand-picked).
- Normalized residual `||Φ(P_t+1) - K Φ(P_t)|| / ||Φ(P_t+1)||`, reported as median + IQR,
  Wilcoxon signed-rank test between class pairs on held-out trajectories.
- **3 random fit/held-out splits** for eDMD (robustness check, motivated by this project's
  earlier n=8-vs-n=20 heterogeneity scare).
- Jacobian sensitivity: single pass, no CV needed (it's a direct measurement, not a fit).

**Budget:** ~40 fit + 20 held-out trajectories/class x 3 splits for eDMD (feature extraction is
forward-pass-only, cheap; eDMD fitting itself is CPU/numpy, no GPU). ~15 trajectories/class,
single pass, for Jacobian sensitivity (backward-pass, the expensive part). Est. 2-3 GPU-hours
total out of this week's 20-hour Kaggle quota.

**Known limitations, stated up front (do not discover these later and treat as new news):**
- This tests whether *this specific pattern* (from A1's 5 tested classes) has a geometric
  correlate — it does not validate the periodic/aperiodic framing as a general theory, since no
  new systems are introduced.
- Lorenz/Rossler/SprottB vs. Burgers/Harmonic differ on multiple axes at once (chaoticity,
  channel count, ODE-vs-PDE origin) — a positive result here cannot by itself say *which* axis
  drives the effect.
- N=20 held-out/class supports directional/significance claims, not precise effect-size
  estimates.


## Section 1 — Setup

In [1]:
# Install panda repo (architecture + training code)
!git clone --depth=1 https://github.com/abao1999/panda.git

# Install panda dependencies
# Note: panda uses uv but we install manually for Kaggle compatibility
%cd panda
!pip install -e . --quiet

# Additional dependencies needed for training
!pip install gluonts wandb --quiet

# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
        print(f'  VRAM: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB')
        print(f'  Compute capability: {torch.cuda.get_device_capability(i)}')

Cloning into 'panda'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 133 (delta 25), reused 47 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 3.77 MiB | 12.98 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/kaggle/working/panda
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 6.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Pre

In [2]:
import subprocess
subprocess.run(['pip', 'uninstall', 'peft', '-y'], capture_output=True)

# Then restart kernel again (skip Cell 1 again after restart)
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

{'status': 'ok', 'restart': True}

In [1]:

import os, sys, json, glob
import numpy as np
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device={device}')

# ADJUST if your Kaggle session clones panda elsewhere
PANDA_REPO_PATH = '/kaggle/working/panda'
if PANDA_REPO_PATH not in sys.path:
    sys.path.insert(0, PANDA_REPO_PATH)

from panda.patchtst.pipeline import PatchTSTPipeline

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)


device=cuda


## Section 2 — Locate and load `baseline_100k` checkpoint

Robust locator (search-by-content, not hardcoded path depth) — same pattern used throughout
this project after being bitten twice by nesting mismatches. Searches for a directory whose
`config.json` has `use_dynamics_embedding: true`, under a dataset-root hint you set below.

In [2]:

BASELINE_100K_DIR = '/kaggle/input/datasets/anujb2/eval-checkpoints/panda-100k-baseline-checkpoint/panda-100k-baseline-checkpoint'
print(f'\nUsing: {BASELINE_100K_DIR}')

pipe = PatchTSTPipeline.from_pretrained(
    mode='predict', pretrain_path=BASELINE_100K_DIR, device_map=device,
)
model = pipe.model
model.eval()
print('Loaded baseline_100k pipeline.')



Using: /kaggle/input/datasets/anujb2/eval-checkpoints/panda-100k-baseline-checkpoint/panda-100k-baseline-checkpoint
Loaded baseline_100k pipeline.


## Section 3 — Locate the dynamics-embedding module and register hooks

We don't hardcode the module path (repo internals may differ from memory) — instead we search
`model.named_modules()` for the lift by class-name / attribute signature, print candidates, and
let you confirm before hooking. The lift's `forward` should take a patch tensor and return the
concatenated [raw | poly | rff] dictionary (Φ_pre) as an intermediate, then a learned linear
layer projects it to d_model (Φ_post). If the module is a single fused block that only exposes
the *final* projected output, we capture the hook's **input** as Φ_pre and **output** as Φ_post
(the input to a `nn.Linear` projection layer, if present, is the pre-projection dictionary).

In [3]:

print('Modules with "dyn" or "embed" or "koopman" or "lift" in their name:\n')
candidate_modules = []
for name, mod in model.named_modules():
    lname = name.lower()
    cname = mod.__class__.__name__.lower()
    if any(k in lname or k in cname for k in ['dynamics', 'koopman', 'lift', 'rff', 'poly', 'dict']):
        candidate_modules.append((name, mod.__class__.__name__))
        print(f'  {name:50s}  {mod.__class__.__name__}')

assert candidate_modules, (
    'No lift-like module found by name search. Print model to inspect manually:\n'
    '  print(model)\n'
    'then set DYNAMICS_EMBED_MODULE_NAME by hand below.'
)


Modules with "dyn" or "embed" or "koopman" or "lift" in their name:

                                                      PatchTSTForPrediction
  head                                                PatchTSTPredictionHead


In [4]:
for name, mod in model.named_modules():
    print(f'{name:60s} {mod.__class__.__name__}  {mod.__class__.__module__}')

                                                             PatchTSTForPrediction  panda.patchtst.patchtst
model                                                        PatchTSTModel  panda.patchtst.patchtst
model.scaler                                                 PatchTSTScaler  transformers.models.patchtst.modeling_patchtst
model.scaler.scaler                                          PatchTSTStdScaler  transformers.models.patchtst.modeling_patchtst
model.patchifier                                             PatchTSTPatchify  panda.patchtst.modules
model.masking                                                Identity  torch.nn.modules.linear
model.encoder                                                PatchTSTEncoder  panda.patchtst.patchtst
model.encoder.embedder                                       PatchTSTKernelEmbedding  panda.patchtst.modules
model.encoder.embedder.projection                            Linear  torch.nn.modules.linear
model.encoder.layers                    

In [6]:
# CONFIRMED against a full named_modules() dump (2026-07-12): everything routes through
# panda.patchtst.* (not stock transformers.models.patchtst), so this IS Panda's real
# architecture -- that concern is resolved.
#
# The original keyword search (['dynamics','koopman','lift','rff','poly','dict']) found
# nothing real -- its only "hit" was a false positive: 'dict' matched inside 'Prediction'
# (PatchTSTForPrediction / PatchTSTPredictionHead), not an actual dictionary/lift module.
# The real lift is named 'embedder' (class PatchTSTKernelEmbedding), which none of those
# keywords would ever match -- confirming the failure was a naming-convention mismatch,
# not a missing module.
#
# Locating it now by structural signature instead of guessed keywords: the embedder is the
# encoder's direct child that contains a nn.Linear named 'projection' and sits before the
# encoder's attention layers.
print('Modules with "embed" or "kernel" (broadened, confirmed-safe keyword set) in their name:\n')
candidate_modules = []
for name, mod in model.named_modules():
    lname = name.lower()
    cname = mod.__class__.__name__.lower()
    if any(k in lname or k in cname for k in ['embed', 'kernel']):
        candidate_modules.append((name, mod.__class__.__name__))
        print(f'  {name:50s}  {mod.__class__.__name__}')

# Confirmed exact path from the printed module dump -- set explicitly rather than trusting
# candidates[0], since 'embed' will also match unrelated things (e.g. positional embeddings,
# if any exist elsewhere in the model).
CONFIRMED_LIFT_MODULE_NAME = 'model.encoder.embedder'

all_module_names = dict(model.named_modules())
assert CONFIRMED_LIFT_MODULE_NAME in all_module_names, (
    f'Expected module "{CONFIRMED_LIFT_MODULE_NAME}" not found in this checkpoint\'s model -- '
    'the architecture may differ from the one inspected in chat. Print the candidate_modules '
    'list above and pick the correct path by hand.'
)
print(f'\nConfirmed lift module present: {CONFIRMED_LIFT_MODULE_NAME} '
      f'({all_module_names[CONFIRMED_LIFT_MODULE_NAME].__class__.__name__})')

Modules with "embed" or "kernel" (broadened, confirmed-safe keyword set) in their name:

  model.encoder.embedder                              PatchTSTKernelEmbedding
  model.encoder.embedder.projection                   Linear

Confirmed lift module present: model.encoder.embedder (PatchTSTKernelEmbedding)


In [7]:
# Using the CONFIRMED path from the previous cell, not a keyword-search guess.
DYNAMICS_EMBED_MODULE_NAME = CONFIRMED_LIFT_MODULE_NAME  # 'model.encoder.embedder'
print(f'Hooking module: {DYNAMICS_EMBED_MODULE_NAME}')

_captured = {'pre': None, 'post': None}

def _hook_fn(module, inputs, output):
    _captured['post'] = output.detach()
    if hasattr(module, 'last_dict'):
        _captured['pre'] = module.last_dict.detach()

target_module = dict(model.named_modules())[DYNAMICS_EMBED_MODULE_NAME]
handle_post = target_module.register_forward_hook(_hook_fn)

inner_linear_name, inner_linear = None, None
for name, mod in target_module.named_modules():
    if isinstance(mod, torch.nn.Linear):
        inner_linear_name, inner_linear = name, mod
        break  # first Linear inside the lift is almost certainly the dict->d_model projection

def _pre_hook_fn(module, inputs):
    _captured['pre'] = inputs[0].detach()

handle_pre = None
if inner_linear is not None:
    handle_pre = inner_linear.register_forward_pre_hook(_pre_hook_fn)
    print(f'Found inner projection Linear at "{DYNAMICS_EMBED_MODULE_NAME}.{inner_linear_name}" '
          f'(in_features={inner_linear.in_features}, out_features={inner_linear.out_features}) '
          '-- hooking its input as Phi_pre.')
    if inner_linear_name != 'projection':
        print('  [NOTE] name differs from the expected "projection" seen in the module dump -- '
              'still likely correct (first Linear inside the embedder), but worth a quick eyeball '
              'against `print(target_module)` if in_features does not look like ~392.')
    elif inner_linear.in_features not in (392,):
        print(f'  [NOTE] in_features={inner_linear.in_features} != the 392 expected from '
              '16 (raw) + 120 (poly) + 256 (rff) in the architecture notes -- if your config\'s '
              'num_poly_feats/num_rff/patch_length differ from those defaults this is fine, '
              'just confirm against `cfg` in Section 4 before trusting Phi_pre downstream.')
else:
    print('[WARNING] No inner nn.Linear found inside the lift module. Phi_pre will only be '
          'populated if the module sets a `.last_dict` attribute during forward(). Inspect '
          'the module source before trusting Phi_pre results:')
    print(target_module)

Hooking module: model.encoder.embedder
Found inner projection Linear at "model.encoder.embedder.projection" (in_features=512, out_features=512) -- hooking its input as Phi_pre.
  [NOTE] in_features=512 != the 392 expected from 16 (raw) + 120 (poly) + 256 (rff) in the architecture notes -- if your config's num_poly_feats/num_rff/patch_length differ from those defaults this is fine, just confirm against `cfg` in Section 4 before trusting Phi_pre downstream.


In [8]:
# 1. Confirm whether raw+poly+rff actually sums to 512 under THIS checkpoint's config
cfg = model.config
patch_length = getattr(cfg, 'patch_length', None)
num_poly_feats = getattr(cfg, 'num_poly_feats', None)
num_rff = getattr(cfg, 'num_rff', None)
print(f'patch_length={patch_length}, num_poly_feats={num_poly_feats}, num_rff={num_rff}')
implied_dict_dim = (patch_length or 0) + (num_poly_feats or 0) + (num_rff or 0)
print(f'implied raw+poly+rff dim = {implied_dict_dim}  (projection.in_features=512)')

# 2. List EVERY submodule inside the embedder, not just the first Linear -- confirms
# whether 'projection' really is the only/final Linear, or whether we stopped early
print('\nFull embedder structure:')
print(target_module)

patch_length=16, num_poly_feats=120, num_rff=256
implied raw+poly+rff dim = 392  (projection.in_features=512)

Full embedder structure:
PatchTSTKernelEmbedding(
  (projection): Linear(in_features=512, out_features=512, bias=False)
)


In [9]:
import inspect
print(inspect.getsource(target_module.__class__))

class PatchTSTKernelEmbedding(nn.Module):
    def __init__(self, config: PatchTSTConfig):
        super().__init__()
        poly_degrees_lst = range(2, 2 + config.poly_degrees)
        # assert (
        #     config.patch_length
        #     + len(poly_degrees_lst) * config.num_poly_feats
        #     + config.num_rff
        #     == config.d_model
        # ), (
        #     f"Sum of features must equal d_model: d_poly + d_rff + patch_length = "
        #     f"{len(poly_degrees_lst) * config.num_poly_feats} + {config.num_rff}"
        #     f" + {config.patch_length} != {config.d_model}"
        # )
        self.num_poly_feats = config.num_poly_feats
        self.patch_indices = [
            torch.randint(
                high=config.patch_length,
                size=(self.num_poly_feats, d),
                requires_grad=False,
            )
            for d in poly_degrees_lst
        ]
        self.freq_weights = nn.Parameter(
            config.rff_scale * torch.randn(co

## Section 4 — Sanity check the hooks with one dummy forward pass

Confirm shapes before running the full extraction loop. **CONFIRMED against the actual
`PatchTSTKernelEmbedding.forward()` source (2026-07-12):** Φ_pre = `cat([x, *poly_feats, rff_feats])`
where `poly_feats` has **one entry per polynomial degree** (`poly_degrees_lst = range(2, 2+config.poly_degrees)`
-- with `poly_degrees=2` this means degrees {2,3}, i.e. TWO chunks of `num_poly_feats` each, not one).
Expected Φ_pre dim = `patch_length + poly_degrees * num_poly_feats + num_rff` -- e.g. for this project's
config (16, 2, 120, 256): `16 + 2*120 + 256 = 512`, matching `projection.in_features`. (Earlier chat
discussion said 392 -- that was wrong, it missed the second polynomial degree; corrected here.)
Φ_post should be `[..., d_model]` (also 512 here, since `projection` is 512→512, no bias).

In [10]:
cfg = model.config
print('Relevant config fields:')
for k in ['use_dynamics_embedding', 'num_poly_feats', 'poly_degrees', 'num_rff',
          'rff_trainable', 'rff_scale', 'patch_length', 'd_model', 'context_length']:
    if hasattr(cfg, k):
        print(f'  {k} = {getattr(cfg, k)}')

patch_length = getattr(cfg, 'patch_length', 16)
poly_degrees = getattr(cfg, 'poly_degrees', 2)
num_poly_feats = getattr(cfg, 'num_poly_feats', 120)
num_rff = getattr(cfg, 'num_rff', 256)
expected_pre_dim = patch_length + poly_degrees * num_poly_feats + num_rff
print(f'\nExpected Phi_pre dim = patch_length + poly_degrees*num_poly_feats + num_rff '
      f'= {patch_length} + {poly_degrees}*{num_poly_feats} + {num_rff} = {expected_pre_dim}')

context_length = getattr(cfg, 'context_length', 512)
n_channels_dummy = 3

# FIXED: previous version called model(dummy) directly with a guessed (batch, C, T) layout,
# which crashed ("maximum size for tensor at dimension 1 is 0 but size is 16") -- channels
# landed in the sequence-length slot, so the patchifier saw a length-3 "sequence" and produced
# zero patches. The repo's own README example builds context as x_context = np.array([...]).T,
# i.e. shape (T, C) time-major (NOT (C, T)), and calls pipe.predict(...), not model(...) directly.
dummy_context = torch.randn(context_length, n_channels_dummy)  # (T, C), matches README convention
with torch.no_grad():
    _ = pipe.predict(
        dummy_context, 32,  # small horizon, just need the hook to fire
        limit_prediction_length=False, sliding_context=True,
    )

print('\nPhi_pre  shape:', None if _captured['pre'] is None else tuple(_captured['pre'].shape))
print('Phi_post shape:', None if _captured['post'] is None else tuple(_captured['post'].shape))

assert _captured['post'] is not None, 'Hook still did not fire -- inspect pipe.predict signature directly (help(pipe.predict)) and adjust the call above.'
if _captured['pre'] is None:
    print('\n[WARNING] Phi_pre not captured. eDMD/Jacobian analysis will be POST-PROJECTION ONLY '
          'until this is fixed.')
elif _captured['pre'].shape[-1] != expected_pre_dim:
    print(f'\n[WARNING] Phi_pre last-dim ({_captured["pre"].shape[-1]}) != expected ({expected_pre_dim}) '
          '-- re-check config fields above against the source before trusting downstream results.')
else:
    print(f'\nPhi_pre dimension matches expectation ({expected_pre_dim}) -- hook confirmed correct.')

print('\n[IMPORTANT] Confirmed input convention: pipe.predict() takes context shaped (T, C) '
      '(time-major), NOT (C, T). Section 5/6 trajectory loaders and extract_patch_features must '
      'produce/consume (T, C) accordingly.')

Relevant config fields:
  use_dynamics_embedding = True
  num_poly_feats = 120
  poly_degrees = 2
  num_rff = 256
  rff_trainable = False
  rff_scale = 1.0
  patch_length = 16
  d_model = 512
  context_length = 512

Expected Phi_pre dim = patch_length + poly_degrees*num_poly_feats + num_rff = 16 + 2*120 + 256 = 512

Phi_pre  shape: (1, 3, 32, 512)
Phi_post shape: (1, 3, 32, 512)

Phi_pre dimension matches expectation (512) -- hook confirmed correct.

[IMPORTANT] Confirmed input convention: pipe.predict() takes context shaped (T, C) (time-major), NOT (C, T). Section 5/6 trajectory loaders and extract_patch_features must produce/consume (T, C) accordingly.


## Section 5 — Trajectory generators (5 classes)

Reuses this project's established generators verbatim where already defined elsewhere
(Burgers: `T=1500, N_x=128, nu=1.0`, PCA to 16 channels, matching Experiment 10/28's protocol).
Lorenz/Rossler/SprottB use the `dysts` library (as used throughout this project's held-out
system evaluation). Harmonic oscillator is a simple closed-form ODE.

**CONFIRM:** paste in the exact `load_burgers_nu1`, Lorenz/Rossler/SprottB trajectory
generators, and `load_harmonic` from `panda_100k_eval_clean.ipynb` / `new_experiments.ipynb`
here rather than reimplementing from scratch — those are already verified against this
project's conventions (fixed IC / RK4 / augmentation choices matter for consistency with A1).
Placeholder stubs below raise `NotImplementedError` until filled in, so this notebook fails
loudly instead of silently running on a different data distribution than A1 used.

In [11]:
from scipy.integrate import solve_ivp
from scipy.fft import fft, ifft, fftfreq
from scipy.linalg import svd

# ============================================================
# All functions below are copied verbatim from panda_100k_eval_clean.ipynb
# (confirmed against the uploaded notebook, 2026-07-12), except where noted.
# All native outputs are (C, T) -- channel-first, this project's established
# convention -- and get transposed to (T, C) at the loader level below, since
# Section 4 confirmed pipe.predict() needs (T, C).
# ============================================================

def simulate_lorenz_gate(n=5000, dt=0.01, sigma=10, rho=28, beta=8/3):
    # Verbatim. NOTE: fixed IC (0.1, 0, 0), NO seed parameter -- deterministic,
    # single orbit. See loader below for how this is handled.
    x, y, z = 0.1, 0.0, 0.0
    xs, ys, zs = [x], [y], [z]
    for _ in range(n - 1):
        k1x = sigma * (y - x); k1y = x * (rho - z) - y; k1z = x * y - beta * z
        k2x = sigma * ((y + dt/2*k1y) - (x + dt/2*k1x))
        k2y = (x + dt/2*k1x) * (rho - (z + dt/2*k1z)) - (y + dt/2*k1y)
        k2z = (x + dt/2*k1x) * (y + dt/2*k1y) - beta * (z + dt/2*k1z)
        k3x = sigma * ((y + dt/2*k2y) - (x + dt/2*k2x))
        k3y = (x + dt/2*k2x) * (rho - (z + dt/2*k2z)) - (y + dt/2*k2y)
        k3z = (x + dt/2*k2x) * (y + dt/2*k2y) - beta * (z + dt/2*k2z)
        k4x = sigma * ((y + dt*k3y) - (x + dt*k3x))
        k4y = (x + dt*k3x) * (rho - (z + dt*k3z)) - (y + dt*k3y)
        k4z = (x + dt*k3x) * (y + dt*k3y) - beta * (z + dt*k3z)
        x += dt/6*(k1x+2*k2x+2*k3x+k4x)
        y += dt/6*(k1y+2*k2y+2*k3y+k4y)
        z += dt/6*(k1z+2*k2z+2*k3z+k4z)
        xs.append(x); ys.append(y); zs.append(z)
    return np.array([xs, ys, zs]).T  # (n, 3)


def simulate_rossler(n_steps=3000, a=0.2, b=0.2, c=5.7, seed=0):
    rng = np.random.default_rng(seed)
    def rhs(t, y):
        return [-y[1]-y[2], y[0]+a*y[1], b+y[2]*(y[0]-c)]
    ic  = rng.standard_normal(3)
    sol = solve_ivp(rhs, [0, n_steps*0.05], ic,
                    t_eval=np.linspace(0, n_steps*0.05, n_steps),
                    method='RK45', rtol=1e-9, atol=1e-9)
    return sol.y  # (3, n_steps)


def simulate_sprott_b(n_steps=3000, seed=0):
    rng = np.random.default_rng(seed)
    def rhs(t, state):
        x, y, z = state
        return [y*z, x - y, 1 - x*y]
    ic  = rng.standard_normal(3)
    sol = solve_ivp(rhs, [0, n_steps*0.05], ic,
                    t_eval=np.linspace(0, n_steps*0.05, n_steps),
                    method='RK45', rtol=1e-9, atol=1e-9)
    return sol.y  # (3, n_steps)


def simulate_burgers_stable(T=1000, N_x=128, nu=0.005, seed=0):
    rng = np.random.default_rng(seed)
    dx  = 2 * np.pi / N_x
    dt_diff   = 0.4 * dx**2 / (2 * nu + 1e-10)
    dt_adv    = 0.4 * dx
    dt        = min(dt_diff, dt_adv, 0.05)
    dt_record = 0.01
    n_sub     = max(1, int(np.ceil(dt_record / dt)))
    dt_act    = dt_record / n_sub
    k       = fftfreq(N_x, d=1.0/N_x).astype(complex)
    dealias = np.abs(k) <= N_x // 3
    L_op    = -nu * k**2
    u0_hat = np.zeros(N_x, dtype=complex)
    for m in range(1, 6):
        amp = rng.standard_normal() + 1j * rng.standard_normal()
        u0_hat[m]       += amp
        u0_hat[N_x - m] += np.conj(amp)
    u0_hat *= dealias
    def rhs_hat(u_hat):
        u_phys = np.real(ifft(u_hat))
        nonlin = fft(0.5 * u_phys**2) * dealias
        return L_op * u_hat - 1j * k * nonlin
    U     = np.zeros((T, N_x), dtype=np.float32)
    u_hat = u0_hat.copy()
    for t in range(T):
        U[t] = np.real(ifft(u_hat)).astype(np.float32)
        for _ in range(n_sub):
            k1    = rhs_hat(u_hat)
            k2    = rhs_hat(u_hat + 0.5*dt_act*k1)
            k3    = rhs_hat(u_hat + 0.5*dt_act*k2)
            k4    = rhs_hat(u_hat +     dt_act*k3)
            u_hat = u_hat + (dt_act/6.0)*(k1+2*k2+2*k3+k4)
            u_hat *= dealias
            if not np.isfinite(u_hat).all():
                return U[:t]
    return U  # (T, N_x)


def pca_reduction(U, n_components):
    U_c  = U - U.mean(axis=0, keepdims=True)
    n_c  = min(n_components, min(U_c.shape)-1)
    _, _, Vt = svd(U_c, full_matrices=False)
    return (U_c @ Vt[:n_c].T).astype(np.float32)  # (T, n_components)


def simulate_harmonic(n_steps=3000, omega=1.0, seed=0):
    rng = np.random.default_rng(seed)
    dt  = 0.05
    x, v = float(rng.standard_normal()), float(rng.standard_normal())
    traj = []
    for _ in range(n_steps):
        traj.append(x)
        x_new = x + v * dt
        v_new = v - omega**2 * x * dt
        x, v  = x_new, v_new
    return np.array(traj, dtype=np.float32)  # (n_steps,)


# ============================================================
# TRAJ_LOADERS -- wrap the above into the (n_traj, length, seed) -> list of
# (T, C) arrays interface Section 6 expects. `length` will always be called
# as context_length (512), per Section 6's exact-match requirement.
# ============================================================

TRANSIENT_DISCARD = 500  # matches A1's convention (e.g. lorenz_traj_gate[500:3500])

def load_lorenz_trajectories(n_traj, length=512, seed=0):
    # NOTE: simulate_lorenz_gate has NO seed/IC parameter -- fixed IC (0.1,0,0),
    # single deterministic orbit (confirmed, discussed explicitly in chat).
    # `seed` is accepted for interface consistency but IGNORED here; documented,
    # not a silent inconsistency. n_traj trajectories are non-overlapping windows
    # sliced from ONE long simulated orbit -- these are correlated with each
    # other (same underlying attractor path), unlike every other class below,
    # which uses genuinely independent random ICs. This is a known, accepted
    # limitation (see chat discussion; option (a) was chosen deliberately).
    n_needed = TRANSIENT_DISCARD + n_traj * length
    full = simulate_lorenz_gate(n=n_needed, dt=0.01)  # (n_needed, 3)
    full = full[TRANSIENT_DISCARD:]
    trajs = [full[i*length:(i+1)*length] for i in range(n_traj)]  # each (length, 3) already T-major
    return trajs

def load_rossler_trajectories(n_traj, length=512, seed=0):
    trajs = []
    for i in range(n_traj):
        n_needed = TRANSIENT_DISCARD + length
        y = simulate_rossler(n_steps=n_needed, seed=seed + i)  # (3, n_needed)
        y = y[:, TRANSIENT_DISCARD:TRANSIENT_DISCARD + length]  # (3, length)
        trajs.append(y.T)  # (length, 3)
    return trajs

def load_sprottb_trajectories(n_traj, length=512, seed=0):
    trajs = []
    for i in range(n_traj):
        n_needed = TRANSIENT_DISCARD + length
        y = simulate_sprott_b(n_steps=n_needed, seed=seed + i)
        y = y[:, TRANSIENT_DISCARD:TRANSIENT_DISCARD + length]
        trajs.append(y.T)
    return trajs

def load_burgers_nu1_trajectories(n_traj, length=512, seed=0):
    # FIXED (2nd time): restrict to top-3 PCA channels. Confirmed via direct
    # per-channel check that channels 5-15 (11 of 16) are essentially frozen
    # (median_ratio ~1e-6) for the full window -- pooling them into the eDMD
    # residual statistic was diluting it with near-dead channels, unrelated to
    # the earlier transient-window issue. Top 3 matches the channel count used
    # by the chaotic ODE classes, making the comparison apples-to-apples.
    trajs = []
    for i in range(n_traj):
        U = simulate_burgers_stable(T=length, N_x=128, nu=1.0, seed=seed + i)
        if U.shape[0] < length:
            raise RuntimeError(f'Burgers trajectory {i} diverged before reaching required length '
                                f'({U.shape[0]} < {length}) -- investigate seed={seed+i}.')
        pca_series = pca_reduction(U, 16)          # (length, 16) -- compute full PCA first
        pca_series = pca_series[:, :3]              # keep only top-3 components (already
                                                      # variance-ordered by SVD)
        trajs.append(pca_series)
    return trajs

def load_harmonic_trajectories(n_traj, length=512, seed=0):
    trajs = []
    for i in range(n_traj):
        n_needed = TRANSIENT_DISCARD + length
        s = simulate_harmonic(n_steps=n_needed, omega=1.0, seed=seed + i)  # (n_needed,)
        s = s[TRANSIENT_DISCARD:TRANSIENT_DISCARD + length]
        trajs.append(s[:, None])  # (length, 1) -- 1 channel, matches A1's OOD convention
    return trajs

TRAJ_LOADERS = {
    'lorenz':   load_lorenz_trajectories,
    'rossler':  load_rossler_trajectories,
    'sprottb':  load_sprottb_trajectories,
    'burgers':  load_burgers_nu1_trajectories,
    'harmonic': load_harmonic_trajectories,
}
CLASS_NAMES = list(TRAJ_LOADERS.keys())
CHAOTIC_CLASSES = ['lorenz', 'rossler', 'sprottb']
NONCHAOTIC_CLASSES = ['burgers', 'harmonic']

print('Trajectory loaders defined (real, verbatim-sourced):', CLASS_NAMES)
print('Channel counts: lorenz=3, rossler=3, sprottb=3, burgers=16, harmonic=1 '
      '-- confirmed confound, see chat discussion before interpreting eDMD results.')

Trajectory loaders defined (real, verbatim-sourced): ['lorenz', 'rossler', 'sprottb', 'burgers', 'harmonic']
Channel counts: lorenz=3, rossler=3, sprottb=3, burgers=16, harmonic=1 -- confirmed confound, see chat discussion before interpreting eDMD results.


## Section 6 — Feature extraction: Φ_pre and Φ_post per patch, per class

For each class: generate `N_FIT + N_HELDOUT` trajectories (60/class for eDMD budget),
run each through the model, collect per-patch (Φ_pre, Φ_post) pairs **in temporal order**
(needed for patch_t -> patch_t+1 pairing in eDMD), save to disk as `.npz` so the CPU-only
eDMD fitting step doesn't need the GPU/model loaded again.

Separately, `N_JACOBIAN` trajectories/class (15/class) are run with `requires_grad=True` for
the Jacobian-sensitivity section.

In [12]:
N_FIT = 40
N_HELDOUT = 20
N_JACOBIAN = 15
TRAJ_LENGTH = context_length  # =512, confirmed in Section 4 -- MUST match model's context window
                               # exactly. Do NOT use a longer length with sliding_context=True:
                               # the hook overwrites _captured on each internal forward pass, so
                               # only the LAST window would be captured, silently losing data.
FEATURE_DIR = './a3_features'
os.makedirs(FEATURE_DIR, exist_ok=True)

def extract_patch_features(traj, model, device):
    """traj: (T, C) with T == context_length exactly, so pipe.predict() does ONE forward
    pass (sliding_context=False) and the hook fires exactly once -- avoids the overwrite bug."""
    x = torch.as_tensor(traj, dtype=torch.float32)
    assert x.shape[0] == context_length, (
        f'traj length {x.shape[0]} != context_length {context_length} -- '
        'sliding_context=False requires an exact match, or the hook will silently only '
        'capture part of the trajectory.'
    )
    with torch.no_grad():
        _ = pipe.predict(x, 32, limit_prediction_length=False, sliding_context=False)
    pre = _captured['pre']
    post = _captured['post']
    pre_np = None if pre is None else pre.squeeze(0).cpu().numpy()   # (C, P, d_pre)
    post_np = post.squeeze(0).cpu().numpy()                          # (C, P, d_post)
    return pre_np, post_np

for cls in CLASS_NAMES:
    loader = TRAJ_LOADERS[cls]
    n_total = N_FIT + N_HELDOUT
    trajs = loader(n_total, length=TRAJ_LENGTH, seed=hash(cls) % (2**31))
    all_pre, all_post, traj_ids, channel_ids = [], [], [], []
    for i, traj in enumerate(trajs):
        pre_np, post_np = extract_patch_features(traj, model, device)
        C = post_np.shape[0]
        for c in range(C):
            all_post.append(post_np[c])
            traj_ids.append(np.full(post_np.shape[1], i))
            channel_ids.append(np.full(post_np.shape[1], c))
            if pre_np is not None:
                all_pre.append(pre_np[c])
    post_arr = np.concatenate(all_post, axis=0)
    ids_arr = np.concatenate(traj_ids, axis=0)
    ch_arr = np.concatenate(channel_ids, axis=0)
    pre_arr = np.concatenate(all_pre, axis=0) if all_pre else None

    save_path = os.path.join(FEATURE_DIR, f'{cls}_features.npz')
    np.savez(save_path, phi_pre=pre_arr, phi_post=post_arr, traj_id=ids_arr,
             channel_id=ch_arr, n_fit=N_FIT, n_heldout=N_HELDOUT)
    print(f'{cls}: saved {save_path}  '
          f'(post shape={post_arr.shape}, pre shape={None if pre_arr is None else pre_arr.shape}, '
          f'{n_total} trajectories x {post_np.shape[0]} channels)')

print('\nFeature extraction complete. GPU/model no longer needed for Sections 7-8.')

lorenz: saved ./a3_features/lorenz_features.npz  (post shape=(5760, 512), pre shape=(5760, 512), 60 trajectories x 3 channels)
rossler: saved ./a3_features/rossler_features.npz  (post shape=(5760, 512), pre shape=(5760, 512), 60 trajectories x 3 channels)
sprottb: saved ./a3_features/sprottb_features.npz  (post shape=(5760, 512), pre shape=(5760, 512), 60 trajectories x 3 channels)
burgers: saved ./a3_features/burgers_features.npz  (post shape=(5760, 512), pre shape=(5760, 512), 60 trajectories x 3 channels)
harmonic: saved ./a3_features/harmonic_features.npz  (post shape=(1920, 512), pre shape=(1920, 512), 60 trajectories x 1 channels)

Feature extraction complete. GPU/model no longer needed for Sections 7-8.


## Section 7 — eDMD fitting (CPU-only, numpy/scipy)

Shared ridge-regularized K, fit on a class-balanced pool across `N_FIT` trajectories/class,
scored on `N_HELDOUT` trajectories/class, repeated over `N_SPLITS` random fit/held-out
resamples. Run separately for Φ_pre and Φ_post — **do not merge the two tables.**

In [13]:
import numpy as np
from scipy.stats import wilcoxon
from sklearn.linear_model import RidgeCV

N_SPLITS = 3
RIDGE_ALPHAS = np.logspace(-4, 3, 15)

def load_class_features(cls, which):  # which in {'phi_pre', 'phi_post'}
    d = np.load(os.path.join(FEATURE_DIR, f'{cls}_features.npz'), allow_pickle=True)
    feats = d[which]
    if feats.dtype == object or feats is None:
        return None, None, None
    return feats, d['traj_id'], d['channel_id']

def make_pairs(feats, traj_id, channel_id):
    """Consecutive-patch pairs (Phi_t, Phi_t+1), grouped by (traj_id, channel_id)
    jointly -- NEVER pairs across a channel boundary (temporal attention, and
    hence any Koopman-linear structure, operates per-channel; confirmed from
    Section 4's captured shape (1, C, P, d_model))."""
    X, Y = [], []
    keys = np.stack([traj_id, channel_id], axis=1)
    unique_keys = np.unique(keys, axis=0)
    for tid, cid in unique_keys:
        mask = (traj_id == tid) & (channel_id == cid)
        f = feats[mask]
        if f.shape[0] < 2:
            continue
        X.append(f[:-1])
        Y.append(f[1:])
    return np.concatenate(X, axis=0), np.concatenate(Y, axis=0)

def fit_and_score(which, seed):
    rng = np.random.default_rng(seed)
    fit_X, fit_Y, held = {}, {}, {}
    for cls in CLASS_NAMES:
        feats, traj_id, channel_id = load_class_features(cls, which)
        if feats is None:
            print(f'[SKIP] {which} not available for class={cls}')
            return None
        traj_ids_unique = np.unique(traj_id)
        rng.shuffle(traj_ids_unique)
        fit_ids = set(traj_ids_unique[:N_FIT])
        held_ids = set(traj_ids_unique[N_FIT:N_FIT + N_HELDOUT])

        fit_mask = np.isin(traj_id, list(fit_ids))
        held_mask = np.isin(traj_id, list(held_ids))

        X_f, Y_f = make_pairs(feats[fit_mask], traj_id[fit_mask], channel_id[fit_mask])
        fit_X[cls], fit_Y[cls] = X_f, Y_f
        held[cls] = (feats[held_mask], traj_id[held_mask], channel_id[held_mask])

    min_pairs = min(fit_X[c].shape[0] for c in CLASS_NAMES)
    Xp, Yp = [], []
    for cls in CLASS_NAMES:
        idx = rng.choice(fit_X[cls].shape[0], size=min_pairs, replace=False)
        Xp.append(fit_X[cls][idx]); Yp.append(fit_Y[cls][idx])
    Xp = np.concatenate(Xp, axis=0); Yp = np.concatenate(Yp, axis=0)

    ridge = RidgeCV(alphas=RIDGE_ALPHAS, alpha_per_target=False)
    ridge.fit(Xp, Yp)
    K = ridge.coef_
    intercept = ridge.intercept_
    print(f'  [{which}, seed={seed}] selected alpha={ridge.alpha_:.4g}, pooled pairs={Xp.shape[0]}')

    rows = []
    for cls in CLASS_NAMES:
        feats_h, traj_id_h, channel_id_h = held[cls]
        Xh, Yh = make_pairs(feats_h, traj_id_h, channel_id_h)
        pred = Xh @ K.T + intercept
        resid = np.linalg.norm(Yh - pred, axis=1) / (np.linalg.norm(Yh, axis=1) + 1e-12)
        rows.append({'class': cls, 'which': which, 'seed': seed,
                      'median_resid': float(np.median(resid)),
                      'iqr_low': float(np.percentile(resid, 25)),
                      'iqr_high': float(np.percentile(resid, 75)),
                      'n_pairs': len(resid), '_raw_resid': resid})
    return rows

all_rows = []
for which in ['phi_pre', 'phi_post']:
    for split_seed in range(N_SPLITS):
        rows = fit_and_score(which, seed=split_seed)
        if rows is not None:
            all_rows.extend(rows)

import pandas as pd
edmd_df = pd.DataFrame([{k: v for k, v in r.items() if k != '_raw_resid'} for r in all_rows])
print('\n=== eDMD residuals (median, IQR) by class / feature-space / split ===')
print(edmd_df.to_string(index=False))
edmd_df.to_csv('a3_edmd_residuals.csv', index=False)

  [phi_pre, seed=0] selected alpha=100, pooled pairs=6200
  [phi_pre, seed=1] selected alpha=31.62, pooled pairs=6200
  [phi_pre, seed=2] selected alpha=100, pooled pairs=6200
  [phi_post, seed=0] selected alpha=31.62, pooled pairs=6200
  [phi_post, seed=1] selected alpha=10, pooled pairs=6200
  [phi_post, seed=2] selected alpha=31.62, pooled pairs=6200

=== eDMD residuals (median, IQR) by class / feature-space / split ===
   class    which  seed  median_resid  iqr_low  iqr_high  n_pairs
  lorenz  phi_pre     0      0.835169 0.522995  1.247374     1860
 rossler  phi_pre     0      0.537316 0.363573  0.750600     1860
 sprottb  phi_pre     0      0.565425 0.393091  0.750980     1860
 burgers  phi_pre     0      0.312201 0.223292  0.447322     1860
harmonic  phi_pre     0      0.439509 0.298062  0.636767      620
  lorenz  phi_pre     1      0.982696 0.591967  1.562712     1860
 rossler  phi_pre     1      0.497301 0.319838  0.754586     1860
 sprottb  phi_pre     1      0.556880 0.39016

In [14]:

# Significance: chaotic classes vs. Burgers, paired Wilcoxon per split/feature-space,
# using the raw per-trajectory-pair residual distributions (not just the medians).
sig_rows = []
for which in ['phi_pre', 'phi_post']:
    for split_seed in range(N_SPLITS):
        subset = [r for r in all_rows if r['which'] == which and r['seed'] == split_seed]
        if not subset:
            continue
        burgers_resid = next((r['_raw_resid'] for r in subset if r['class'] == 'burgers'), None)
        if burgers_resid is None:
            continue
        for chaotic_cls in CHAOTIC_CLASSES:
            chaotic_resid = next((r['_raw_resid'] for r in subset if r['class'] == chaotic_cls), None)
            if chaotic_resid is None:
                continue
            n = min(len(burgers_resid), len(chaotic_resid))
            try:
                stat, p = wilcoxon(chaotic_resid[:n], burgers_resid[:n])
            except ValueError:
                p = float('nan')
            sig_rows.append({'which': which, 'seed': split_seed, 'class': chaotic_cls,
                              'compared_to': 'burgers',
                              'median_diff': float(np.median(chaotic_resid) - np.median(burgers_resid)),
                              'wilcoxon_p': p})

sig_df = pd.DataFrame(sig_rows)
print('=== Chaotic vs. Burgers residual comparison ===')
print(sig_df.to_string(index=False))
sig_df.to_csv('a3_edmd_significance.csv', index=False)
print('\n[REMINDER] positive median_diff = chaotic class has HIGHER (worse) residual than Burgers, '
      'i.e. consistent with the A1-motivated hypothesis. Check consistency ACROSS all 3 splits '
      'before treating any single split as confirmatory.')


=== Chaotic vs. Burgers residual comparison ===
   which  seed   class compared_to  median_diff    wilcoxon_p
 phi_pre     0  lorenz     burgers     0.522968 1.439201e-219
 phi_pre     0 rossler     burgers     0.225114 8.494958e-126
 phi_pre     0 sprottb     burgers     0.253224 2.820944e-148
 phi_pre     1  lorenz     burgers     0.677210 8.019218e-241
 phi_pre     1 rossler     burgers     0.191816 2.464627e-109
 phi_pre     1 sprottb     burgers     0.251394 9.256706e-168
 phi_pre     2  lorenz     burgers     0.618182 1.115909e-234
 phi_pre     2 rossler     burgers     0.188040 2.965317e-106
 phi_pre     2 sprottb     burgers     0.239473 6.281727e-159
phi_post     0  lorenz     burgers     0.405333 6.873969e-240
phi_post     0 rossler     burgers     0.155076 1.132182e-119
phi_post     0 sprottb     burgers     0.223949 3.165528e-191
phi_post     1  lorenz     burgers     0.447683 4.197422e-241
phi_post     1 rossler     burgers     0.150091 3.399280e-107
phi_post     1 sprottb

In [17]:
def persistence_baseline_residual(feats, traj_id, channel_id):
    """K = Identity: predicts no change patch-to-patch. If this alone scores
    low on Burgers, low eDMD residual there is NOT evidence of Koopman
    linearization -- it just means the signal is slowly varying."""
    X, Y = make_pairs(feats, traj_id, channel_id)
    resid = np.linalg.norm(Y - X, axis=1) / (np.linalg.norm(Y, axis=1) + 1e-12)
    return resid

print(f'{"class":10s} {"which":10s} {"persistence_median":>20s} {"fitted_K_median (seed0)":>25s}')
for which in ['phi_pre', 'phi_post']:
    for cls in CLASS_NAMES:
        feats, traj_id, channel_id = load_class_features(cls, which)
        resid = persistence_baseline_residual(feats, traj_id, channel_id)
        fitted_median = edmd_df[(edmd_df['class']==cls) & (edmd_df['which']==which) &
                                 (edmd_df['seed']==0)]['median_resid'].values[0]
        print(f'{cls:10s} {which:10s} {np.median(resid):>20.4f} {fitted_median:>25.4f}')

class      which        persistence_median   fitted_K_median (seed0)
lorenz     phi_pre                  1.2154                    0.8383
rossler    phi_pre                  1.0047                    0.4882
sprottb    phi_pre                  0.9109                    0.5372
burgers    phi_pre                  0.0000                    0.0951
harmonic   phi_pre                  1.2113                    0.4431
lorenz     phi_post                 1.2352                    0.6888
rossler    phi_post                 1.0332                    0.4569
sprottb    phi_post                 0.9185                    0.5240
burgers    phi_post                 0.0000                    0.0990
harmonic   phi_post                 1.2138                    0.4221


In [18]:
print(f'{"class":10s} {"mean patch-to-patch |Y-X|":>28s} {"mean |Y| (signal scale)":>25s} {"ratio":>10s}')
for cls in CLASS_NAMES:
    feats, traj_id, channel_id = load_class_features(cls, 'phi_post')
    X, Y = make_pairs(feats, traj_id, channel_id)
    delta = np.linalg.norm(Y - X, axis=1).mean()
    scale = np.linalg.norm(Y, axis=1).mean()
    print(f'{cls:10s} {delta:>28.6f} {scale:>25.6f} {delta/scale:>10.6f}')

# Also: does the raw Burgers PCA signal itself vary across the 512-step window,
# or has it already decayed to near-constant by the time extraction starts?
sample_traj = load_burgers_nu1_trajectories(1, length=512, seed=999)[0]  # (512, 16)
print('\nBurgers PCA-component RMS across the 512-step window (per channel):')
print(sample_traj.std(axis=0))
print('Burgers PCA-component values, first vs last 5 steps (channel 0):')
print('first:', sample_traj[:5, 0])
print('last: ', sample_traj[-5:, 0])

class         mean patch-to-patch |Y-X|   mean |Y| (signal scale)      ratio
lorenz                        16.913763                 13.301966   1.271524
rossler                       17.600014                 14.689687   1.198120
sprottb                       13.238275                 13.464170   0.983222
burgers                        0.449638                  7.776350   0.057821
harmonic                      14.526838                 13.052001   1.112997

Burgers PCA-component RMS across the 512-step window (per channel):
[2.1164108e-03 2.9211184e-03 1.1287566e-03 2.2475455e-04 3.4072989e-05
 6.0260084e-07 9.0928370e-07 2.2871782e-07 1.8189219e-07 1.2225452e-08
 6.2564953e-09 4.0692960e-10 1.1604715e-09 3.0616018e-10 4.1764806e-10
 1.4420318e-10]
Burgers PCA-component values, first vs last 5 steps (channel 0):
first: [0.00806803 0.00815599 0.00824299 0.00832905 0.00841419]
last:  [0.01664931 0.01664984 0.01665037 0.01665089 0.01665141]


In [23]:
sample_traj = load_burgers_nu1_trajectories(1, length=512, seed=999)[0]  # (512, 16)
print('Burgers PCA-component RMS across the full 512-step window (from t=0 now):')
print(sample_traj.std(axis=0))
print('\nFirst vs last 5 steps, channel 0:')
print('first:', sample_traj[:5, 0])
print('last: ', sample_traj[-5:, 0])

# And re-run the patch-to-patch delta/scale check on the freshly-saved features:
print('\nPersistence check on saved features (confirms whether Section 6 actually re-ran):')
for cls in ['burgers']:
    feats, traj_id, channel_id = load_class_features(cls, 'phi_post')
    X, Y = make_pairs(feats, traj_id, channel_id)
    delta = np.linalg.norm(Y - X, axis=1).mean()
    scale = np.linalg.norm(Y, axis=1).mean()
    print(f'{cls}: mean|Y-X|={delta:.6f}, mean|Y|={scale:.6f}, ratio={delta/scale:.6f}')

Burgers PCA-component RMS across the full 512-step window (from t=0 now):
[4.7533419e-02 1.5831111e-02 7.3944042e-03 1.3882897e-03 2.0158001e-04
 4.8118923e-06 1.6548555e-06 6.9966381e-07 3.1395385e-07 5.3971878e-08
 1.7327139e-08 2.6485187e-09 4.0486525e-09 3.0977885e-09 1.9268243e-09
 1.8761643e-09]

First vs last 5 steps, channel 0:
first: [-0.29627907 -0.27915964 -0.26389828 -0.25017968 -0.2377516 ]
last:  [0.02237445 0.0223782  0.02238191 0.02238559 0.02238922]

Persistence check on saved features (confirms whether Section 6 actually re-ran):
burgers: mean|Y-X|=1.855348, mean|Y|=8.046760, ratio=0.230571


In [24]:
print(f'{"class":10s} {"which":10s} {"persistence_median":>20s} {"fitted_K_median (seed0)":>25s}')
for which in ['phi_pre', 'phi_post']:
    for cls in ['burgers']:
        feats, traj_id, channel_id = load_class_features(cls, which)
        resid = persistence_baseline_residual(feats, traj_id, channel_id)
        fitted_median = edmd_df[(edmd_df['class']==cls) & (edmd_df['which']==which) &
                                 (edmd_df['seed']==0)]['median_resid'].values[0]
        print(f'{cls:10s} {which:10s} {np.median(resid):>20.4f} {fitted_median:>25.4f}')

class      which        persistence_median   fitted_K_median (seed0)
burgers    phi_pre                  0.0001                    0.0972
burgers    phi_post                 0.0001                    0.1048


In [25]:
def residual_by_patch_position(feats, traj_id, channel_id, K=None, intercept=None):
    """Residual (persistence, K=Identity) as a function of position-within-trajectory,
    averaged across trajectories/channels -- shows whether large residuals cluster early."""
    positions = []
    resids = []
    for tid, cid in np.unique(np.stack([traj_id, channel_id], axis=1), axis=0):
        mask = (traj_id == tid) & (channel_id == cid)
        f = feats[mask]
        if f.shape[0] < 2:
            continue
        delta = np.linalg.norm(f[1:] - f[:-1], axis=1) / (np.linalg.norm(f[1:], axis=1) + 1e-12)
        resids.append(delta)
        positions.append(np.arange(len(delta)))
    positions = np.concatenate(positions)
    resids = np.concatenate(resids)
    df_pos = pd.DataFrame({'patch_position': positions, 'residual': resids})
    return df_pos.groupby('patch_position')['residual'].median()

feats, traj_id, channel_id = load_class_features('burgers', 'phi_post')
profile = residual_by_patch_position(feats, traj_id, channel_id)
print('Burgers persistence residual by within-trajectory patch position (0-30):')
print(profile.to_string())

Burgers persistence residual by within-trajectory patch position (0-30):
patch_position
0     0.002339
1     0.001920
2     0.001219
3     0.000652
4     0.000561
5     0.000337
6     0.000213
7     0.000166
8     0.000166
9     0.000147
10    0.000118
11    0.000086
12    0.000053
13    0.000032
14    0.000022
15    0.000036
16    0.000046
17    0.000054
18    0.000056
19    0.000055
20    0.000051
21    0.000046
22    0.000041
23    0.000037
24    0.000033
25    0.000029
26    0.000025
27    0.000022
28    0.000019
29    0.000017
30    0.000014


In [26]:
def residual_by_channel(feats, traj_id, channel_id):
    X, Y = make_pairs(feats, traj_id, channel_id)
    return None  # placeholder, real logic below uses raw feats/ids directly

feats, traj_id, channel_id = load_class_features('burgers', 'phi_post')
rows = []
for cid in np.unique(channel_id):
    mask = channel_id == cid
    f, tid = feats[mask], traj_id[mask]
    X, Y = make_pairs(f, tid, np.zeros_like(tid))  # already single-channel slice
    delta = np.linalg.norm(Y - X, axis=1)
    scale = np.linalg.norm(Y, axis=1)
    rows.append({'channel': cid, 'mean_delta': delta.mean(), 'mean_scale': scale.mean(),
                 'median_ratio': np.median(delta / (scale + 1e-12))})
pd.DataFrame(rows).to_string(index=False)
print(pd.DataFrame(rows).to_string(index=False))

 channel  mean_delta  mean_scale  median_ratio
       0    8.676756   10.114580  8.787076e-02
       1    8.508792   10.649456  1.948003e-01
       2    8.316073    9.742036  2.081167e-01
       3    3.155106    8.122830  1.058457e-01
       4    0.969403    7.549026  1.709585e-02
       5    0.035840    7.506374  7.308872e-04
       6    0.016575    7.506371  4.336960e-04
       7    0.004666    7.506390  9.878394e-05
       8    0.001636    7.506386  4.324768e-05
       9    0.000504    7.506386  1.283607e-05
      10    0.000099    7.506386  2.071459e-06
      11    0.000041    7.506386  1.032850e-06
      12    0.000035    7.506386  1.339986e-06
      13    0.000013    7.506386  6.524022e-07
      14    0.000010    7.506386  5.763223e-07
      15    0.000009    7.506386  5.395696e-07


In [31]:
print(f'{"class":10s} {"which":10s} {"persistence_median":>20s} {"fitted_K_median (seed0)":>25s}')
for which in ['phi_pre', 'phi_post']:
    for cls in ['burgers']:
        feats, traj_id, channel_id = load_class_features(cls, which)
        resid = persistence_baseline_residual(feats, traj_id, channel_id)
        fitted_median = edmd_df[(edmd_df['class']==cls) & (edmd_df['which']==which) &
                                 (edmd_df['seed']==0)]['median_resid'].values[0]
        print(f'{cls:10s} {which:10s} {np.median(resid):>20.4f} {fitted_median:>25.4f}')

class      which        persistence_median   fitted_K_median (seed0)
burgers    phi_pre                  0.1555                    0.2973
burgers    phi_post                 0.1616                    0.3143


In [32]:
print(f'{"class":10s} {"mean patch-to-patch |Y-X|":>28s} {"mean |Y|":>12s} {"mean_ratio":>12s} {"median_resid (fitted K, phi_post, seed0)":>42s}')
for cls in CLASS_NAMES:
    feats, traj_id, channel_id = load_class_features(cls, 'phi_post')
    X, Y = make_pairs(feats, traj_id, channel_id)
    delta = np.linalg.norm(Y - X, axis=1).mean()
    scale = np.linalg.norm(Y, axis=1).mean()
    fitted_median = edmd_df[(edmd_df['class']==cls) & (edmd_df['which']=='phi_post') &
                             (edmd_df['seed']==0)]['median_resid'].values[0]
    print(f'{cls:10s} {delta:>28.6f} {scale:>12.6f} {delta/scale:>12.6f} {fitted_median:>42.4f}')

class         mean patch-to-patch |Y-X|     mean |Y|   mean_ratio   median_resid (fitted K, phi_post, seed0)
lorenz                        16.913763    13.301966     1.271524                                     0.7027
rossler                       17.600014    14.689687     1.198120                                     0.4572
sprottb                       13.238275    13.464170     0.983222                                     0.5186
burgers                        8.500541    10.168691     0.835952                                     0.3143
harmonic                      14.526838    13.052001     1.112997                                     0.4352


## Section 8 — Jacobian sensitivity of the lift

Measures how much the lift itself amplifies small input perturbations, per class
(`N_JACOBIAN` trajectories/class, single pass — no CV needed, this is a direct measurement).
Uses `torch.autograd.functional.jacobian` (or a vector-Jacobian-product loop if the full
Jacobian is too large per patch) on the pre-projection dictionary Φ_pre w.r.t. the input
patch.

In [12]:
# Quick sanity check: does gradient actually flow through pipe.predict()?
test_traj = load_lorenz_trajectories(1, length=context_length, seed=0)[0]  # (512, 3)
x_test = torch.as_tensor(test_traj, dtype=torch.float32).requires_grad_(True)
_ = pipe.predict(x_test, 32, limit_prediction_length=False, sliding_context=False)
phi_test = _captured['pre'] if _captured['pre'] is not None else _captured['post']
scalar = phi_test.sum()
try:
    g, = torch.autograd.grad(scalar, x_test, retain_graph=True)
    print('Gradient flows through pipe.predict(). grad shape:', g.shape,
          'nonzero entries:', (g.abs() > 1e-8).sum().item(), '/', g.numel())
except RuntimeError as e:
    print('[BLOCKED] Gradient does NOT flow through pipe.predict():', e)
    print('Will need to call the underlying model/submodules directly instead, bypassing pipe.predict.')

[BLOCKED] Gradient does NOT flow through pipe.predict(): element 0 of tensors does not require grad and does not have a grad_fn
Will need to call the underlying model/submodules directly instead, bypassing pipe.predict.


In [13]:
import inspect
print(inspect.getsource(type(pipe).predict))

    @torch.no_grad()
    def predict(
        self,
        context: torch.Tensor | list[torch.Tensor],
        prediction_length: int,
        limit_prediction_length: bool = True,
        sliding_context: bool = False,
        verbose: bool = True,
    ) -> torch.Tensor:
        """
        Generate an autoregressive forecast for a given context timeseries

        Parameters
        ----------
        context
            Input series. This is either a 1D tensor, or a list
            of 1D tensors, or a 2D tensor whose first dimension
            is sequence length. In the latter case, use left-padding with
            ``torch.nan`` to align series of different lengths.
        prediction_length
            Time steps to predict. Defaults to what specified
            in ``self.model.config``.
        limit_prediction_length
            Force prediction length smaller or equal than the
            built-in prediction length from the model. True by
            default. When true, fai

In [14]:
import inspect
print("=== PatchTSTModel.forward signature ===")
print(inspect.signature(type(pipe.model.model).forward))
print()
print(inspect.getsource(type(pipe.model.model).forward))
print()
print("=== _prepare_and_validate_context ===")
print(inspect.getsource(type(pipe)._prepare_and_validate_context))

=== PatchTSTModel.forward signature ===
(self, past_values: torch.Tensor, past_observed_mask: torch.Tensor | None = None, future_values: torch.Tensor | None = None, output_hidden_states: bool | None = None, output_attentions: bool | None = None, channel_attention_mask: torch.Tensor | None = None, return_dict: bool | None = None) -> tuple | transformers.models.patchtst.modeling_patchtst.PatchTSTModelOutput

    def forward(
        self,
        past_values: torch.Tensor,
        past_observed_mask: torch.Tensor | None = None,
        future_values: torch.Tensor | None = None,
        output_hidden_states: bool | None = None,
        output_attentions: bool | None = None,
        channel_attention_mask: torch.Tensor | None = None,
        return_dict: bool | None = None,
    ) -> tuple | PatchTSTModelOutput:
        r"""
        Parameters:
            past_values (`torch.Tensor` of shape `(bs, sequence_length, num_input_channels)`, *required*):
                Input sequence to the mod

In [15]:
print(inspect.getsource(type(pipe.model.model.scaler.scaler)))

class PatchTSTStdScaler(nn.Module):
    """
    Standardize features by calculating the mean and scaling along the first dimension, and then normalizes it by
    subtracting from the mean and dividing by the standard deviation.
    """

    def __init__(self, config: PatchTSTConfig):
        super().__init__()
        self.dim = config.scaling_dim if hasattr(config, "scaling_dim") else 1
        self.keepdim = config.keepdim if hasattr(config, "keepdim") else True
        self.minimum_scale = config.minimum_scale if hasattr(config, "minimum_scale") else 1e-5

    def forward(
        self, data: torch.Tensor, observed_indicator: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Parameters:
            data (`torch.Tensor` of shape `(batch_size, sequence_length, num_input_channels)`):
                input for Batch norm calculation
            observed_indicator (`torch.BoolTensor` of shape `(batch_size, sequence_length, num_input_channels)`):
 

In [17]:
# Separate, non-detaching capture for Jacobian work -- the shared _captured hooks
# from Section 3 call .detach() on purpose (correct for Sections 4/6/7's plain
# feature extraction), which would silently break the autograd graph needed here.
_captured_grad = {'pre': None, 'post': None}

def _hook_fn_grad(module, inputs, output):
    _captured_grad['post'] = output  # NOT detached

def _pre_hook_fn_grad(module, inputs):
    _captured_grad['pre'] = inputs[0]  # NOT detached

handle_post_grad = target_module.register_forward_hook(_hook_fn_grad)
handle_pre_grad = None
if inner_linear is not None:
    handle_pre_grad = inner_linear.register_forward_pre_hook(_pre_hook_fn_grad)

def patch_lift_jacobian_norm(model_inner, x_context, patch_idx=0):
    """x_context: (context_length, C), on model's device, requires_grad=True.
    Full-context Jacobian: PatchTSTStdScaler normalizes over the FULL context
    window (confirmed: dim=1=sequence_length) before patching, so every patch's
    Phi genuinely depends on the whole 512-step input via shared loc/scale --
    restricting to a local 16-step window would exclude real signal."""
    x_batched = x_context.unsqueeze(0)
    _ = model_inner(past_values=x_batched)
    phi = _captured_grad['pre'] if _captured_grad['pre'] is not None else _captured_grad['post']
    phi_patch = phi[0, :, patch_idx, :]
    phi_flat = phi_patch.reshape(-1)

    J = torch.zeros(phi_flat.shape[0], x_context.numel())
    for i in range(phi_flat.shape[0]):
        grad_out = torch.zeros_like(phi_flat)
        grad_out[i] = 1.0
        g, = torch.autograd.grad(phi_flat, x_context, grad_outputs=grad_out, retain_graph=True)
        J[i] = g.reshape(-1).cpu()

    T, C = x_context.shape
    patch_length = getattr(model_inner.config, 'patch_length', 16)
    J_reshaped = J.reshape(phi_flat.shape[0], T, C)
    start, end = patch_idx * patch_length, patch_idx * patch_length + patch_length
    local_mask = torch.zeros(T, dtype=torch.bool); local_mask[start:end] = True
    local_grad_mass = J_reshaped[:, local_mask, :].abs().sum().item()
    global_grad_mass = J_reshaped[:, ~local_mask, :].abs().sum().item()

    fro_norm  = torch.linalg.norm(J, ord='fro').item()
    spec_norm = torch.linalg.matrix_norm(J, ord=2).item()
    return fro_norm, spec_norm, local_grad_mass, global_grad_mass

jac_rows = []
for cls in CLASS_NAMES:
    loader = TRAJ_LOADERS[cls]
    trajs = loader(N_JACOBIAN, length=context_length, seed=(hash(cls) + 1) % (2**31))
    for traj in trajs:
        x = torch.as_tensor(traj, dtype=torch.float32, device=device).requires_grad_(True)
        fro, spec, local_mass, global_mass = patch_lift_jacobian_norm(pipe.model.model, x)
        jac_rows.append({'class': cls, 'fro_norm': fro, 'spectral_norm': spec,
                          'local_grad_mass': local_mass, 'global_grad_mass': global_mass,
                          'global_frac': global_mass / (local_mass + global_mass)})

jac_df = pd.DataFrame(jac_rows)
print(jac_df.groupby('class')[['fro_norm', 'spectral_norm', 'global_frac']].median())
jac_df.to_csv('a3_jacobian_sensitivity.csv', index=False)

handle_post_grad.remove()
if handle_pre_grad is not None:
    handle_pre_grad.remove()

              fro_norm  spectral_norm  global_frac
class                                             
burgers   33180.476562   11978.128906     0.523520
harmonic     16.231747       4.949382     0.240592
lorenz       11.828674       2.356946     0.269421
rossler      23.318983       5.865306     0.288228
sprottb      80.431961      17.646322     0.319234


In [18]:
_captured_grad = {'pre': None, 'post': None, 'scaled_input': None}

def _hook_fn_grad(module, inputs, output):
    _captured_grad['post'] = output

def _pre_hook_fn_grad(module, inputs):
    _captured_grad['pre'] = inputs[0]

# NEW: capture the scaler's OUTPUT (= patchifier's INPUT) without detaching --
# this is what the model actually differentiates through internally; raw past_values
# scale is an arbitrary preprocessing artifact (1/scale term), not real sensitivity.
def _scaled_input_hook(module, inputs):
    _captured_grad['scaled_input'] = inputs[0]

handle_post_grad = target_module.register_forward_hook(_hook_fn_grad)
handle_pre_grad = None
if inner_linear is not None:
    handle_pre_grad = inner_linear.register_forward_pre_hook(_pre_hook_fn_grad)
handle_scaled_grad = pipe.model.model.patchifier.register_forward_pre_hook(_scaled_input_hook)

def patch_lift_jacobian_norm(model_inner, x_context, patch_idx=0):
    """Differentiates Phi w.r.t. the SCALED input (post-normalization), not raw
    past_values -- removes the 1/scale confound from PatchTSTStdScaler (confirmed:
    scaled = (x - loc) / scale, scale = per-window std, so classes with small raw
    signal variance would otherwise show mechanically inflated gradients)."""
    x_batched = x_context.unsqueeze(0)
    _ = model_inner(past_values=x_batched)
    phi = _captured_grad['pre'] if _captured_grad['pre'] is not None else _captured_grad['post']
    scaled_input = _captured_grad['scaled_input']  # (1, T, C), non-leaf, in-graph
    phi_patch = phi[0, :, patch_idx, :]
    phi_flat = phi_patch.reshape(-1)

    J = torch.zeros(phi_flat.shape[0], scaled_input.numel())
    for i in range(phi_flat.shape[0]):
        grad_out = torch.zeros_like(phi_flat)
        grad_out[i] = 1.0
        g, = torch.autograd.grad(phi_flat, scaled_input, grad_outputs=grad_out, retain_graph=True)
        J[i] = g.reshape(-1).cpu()

    fro_norm  = torch.linalg.norm(J, ord='fro').item()
    spec_norm = torch.linalg.matrix_norm(J, ord=2).item()
    return fro_norm, spec_norm

jac_rows = []
for cls in CLASS_NAMES:
    loader = TRAJ_LOADERS[cls]
    trajs = loader(N_JACOBIAN, length=context_length, seed=(hash(cls) + 1) % (2**31))
    for traj in trajs:
        x = torch.as_tensor(traj, dtype=torch.float32, device=device).requires_grad_(True)
        fro, spec = patch_lift_jacobian_norm(pipe.model.model, x)
        jac_rows.append({'class': cls, 'fro_norm': fro, 'spectral_norm': spec})

jac_df = pd.DataFrame(jac_rows)
print(jac_df.groupby('class')[['fro_norm', 'spectral_norm']].median())
jac_df.to_csv('a3_jacobian_sensitivity.csv', index=False)

handle_post_grad.remove()
if handle_pre_grad is not None:
    handle_pre_grad.remove()
handle_scaled_grad.remove()

            fro_norm  spectral_norm
class                              
burgers   511.995483     183.296707
harmonic   45.223740      14.371065
lorenz     82.956108      16.551392
rossler    87.175476      17.987686
sprottb    90.845322      17.535614


In [19]:
def jacobian_norm_multi_position(model_inner, x_context, patch_indices):
    x_batched = x_context.unsqueeze(0)
    _ = model_inner(past_values=x_batched)
    phi = _captured_grad['pre'] if _captured_grad['pre'] is not None else _captured_grad['post']
    scaled_input = _captured_grad['scaled_input']
    results = {}
    for patch_idx in patch_indices:
        phi_patch = phi[0, :, patch_idx, :]
        phi_flat = phi_patch.reshape(-1)
        J = torch.zeros(phi_flat.shape[0], scaled_input.numel())
        for i in range(phi_flat.shape[0]):
            grad_out = torch.zeros_like(phi_flat)
            grad_out[i] = 1.0
            g, = torch.autograd.grad(phi_flat, scaled_input, grad_outputs=grad_out, retain_graph=True)
            J[i] = g.reshape(-1).cpu()
        results[patch_idx] = torch.linalg.norm(J, ord='fro').item()
    return results

TEST_POSITIONS = [0, 5, 10, 15, 20, 25, 30]
position_rows = []
for cls in CLASS_NAMES:
    loader = TRAJ_LOADERS[cls]
    trajs = loader(5, length=context_length, seed=(hash(cls) + 99) % (2**31))  # small N, just diagnostic
    for traj in trajs:
        x = torch.as_tensor(traj, dtype=torch.float32, device=device).requires_grad_(True)
        res = jacobian_norm_multi_position(pipe.model.model, x, TEST_POSITIONS)
        for pos, val in res.items():
            position_rows.append({'class': cls, 'patch_position': pos, 'fro_norm': val})

pos_df = pd.DataFrame(position_rows)
print(pos_df.groupby(['class', 'patch_position'])['fro_norm'].median().unstack('patch_position'))

patch_position         0          5          10         15         20  \
class                                                                   
burgers         44.483448  47.164925  57.699543  47.989201  44.900265   
harmonic        44.483448  47.164925  57.699543  47.989201  44.900265   
lorenz          44.483448  47.164925  57.699543  47.989201  44.900265   
rossler         44.483448  47.164925  57.699543  47.989201  44.900265   
sprottb         44.483448  47.164925  57.699543  47.989201  44.900265   

patch_position         25         30  
class                                 
burgers         66.158409  76.090973  
harmonic        66.158409  76.090973  
lorenz          66.158409  76.090973  
rossler         66.158409  76.090973  
sprottb         66.158409  76.090973  


In [20]:
# Hooks were removed at the end of the previous cell -- re-register before this
# diagnostic, or _captured_grad silently goes stale (exactly what just happened:
# identical values across all classes, since no new forward pass was actually
# being captured).
_captured_grad = {'pre': None, 'post': None, 'scaled_input': None}

def _hook_fn_grad(module, inputs, output):
    _captured_grad['post'] = output

def _pre_hook_fn_grad(module, inputs):
    _captured_grad['pre'] = inputs[0]

def _scaled_input_hook(module, inputs):
    _captured_grad['scaled_input'] = inputs[0]

handle_post_grad = target_module.register_forward_hook(_hook_fn_grad)
handle_pre_grad = None
if inner_linear is not None:
    handle_pre_grad = inner_linear.register_forward_pre_hook(_pre_hook_fn_grad)
handle_scaled_grad = pipe.model.model.patchifier.register_forward_pre_hook(_scaled_input_hook)

def jacobian_norm_multi_position(model_inner, x_context, patch_indices):
    x_batched = x_context.unsqueeze(0)
    _ = model_inner(past_values=x_batched)
    phi = _captured_grad['pre'] if _captured_grad['pre'] is not None else _captured_grad['post']
    scaled_input = _captured_grad['scaled_input']
    results = {}
    for patch_idx in patch_indices:
        phi_patch = phi[0, :, patch_idx, :]
        phi_flat = phi_patch.reshape(-1)
        J = torch.zeros(phi_flat.shape[0], scaled_input.numel())
        for i in range(phi_flat.shape[0]):
            grad_out = torch.zeros_like(phi_flat)
            grad_out[i] = 1.0
            g, = torch.autograd.grad(phi_flat, scaled_input, grad_outputs=grad_out, retain_graph=True)
            J[i] = g.reshape(-1).cpu()
        results[patch_idx] = torch.linalg.norm(J, ord='fro').item()
    return results

TEST_POSITIONS = [0, 5, 10, 15, 20, 25, 30]
position_rows = []
for cls in CLASS_NAMES:
    loader = TRAJ_LOADERS[cls]
    trajs = loader(5, length=context_length, seed=(hash(cls) + 99) % (2**31))
    for traj in trajs:
        x = torch.as_tensor(traj, dtype=torch.float32, device=device).requires_grad_(True)
        res = jacobian_norm_multi_position(pipe.model.model, x, TEST_POSITIONS)
        for pos, val in res.items():
            position_rows.append({'class': cls, 'patch_position': pos, 'fro_norm': val})

pos_df = pd.DataFrame(position_rows)
print(pos_df.groupby(['class', 'patch_position'])['fro_norm'].median().unstack('patch_position'))

handle_post_grad.remove()
if handle_pre_grad is not None:
    handle_pre_grad.remove()
handle_scaled_grad.remove()

patch_position          0          5          10          15         20  \
class                                                                     
burgers         523.448608  92.142593  77.460098   77.171326  77.862015   
harmonic         47.544682  48.508236  46.806446   46.468494  61.145069   
lorenz           80.559280  77.857635  78.404541   82.289833  92.681808   
rossler          78.979401  85.946449  94.554070  106.728447  78.708916   
sprottb          84.153595  84.343117  97.446777   81.074860  83.913864   

patch_position         25         30  
class                                 
burgers         78.412735  78.724907  
harmonic        68.475548  51.339520  
lorenz          78.200043  79.932953  
rossler         86.827133  83.797180  
sprottb         79.303497  78.111900  


In [21]:

# Significance for Jacobian norms, same chaotic-vs-Burgers comparison as Section 7
from scipy.stats import mannwhitneyu

jac_sig_rows = []
burgers_fro = jac_df[jac_df['class'] == 'burgers']['fro_norm'].values
for chaotic_cls in CHAOTIC_CLASSES:
    chaotic_fro = jac_df[jac_df['class'] == chaotic_cls]['fro_norm'].values
    if len(burgers_fro) and len(chaotic_fro):
        stat, p = mannwhitneyu(chaotic_fro, burgers_fro, alternative='two-sided')
        jac_sig_rows.append({'class': chaotic_cls, 'compared_to': 'burgers',
                              'median_diff': float(np.median(chaotic_fro) - np.median(burgers_fro)),
                              'mannwhitney_p': p})
jac_sig_df = pd.DataFrame(jac_sig_rows)
print(jac_sig_df.to_string(index=False))
jac_sig_df.to_csv('a3_jacobian_significance.csv', index=False)


  class compared_to  median_diff  mannwhitney_p
 lorenz     burgers  -429.039375       0.000003
rossler     burgers  -424.820007       0.000003
sprottb     burgers  -421.150162       0.000003


## Section 9 — Diagnostic companions: effective rank + condition number

Cheap, forward-pass-only, descriptive context for Sections 7-8 — **not** used standalone as
evidence for/against the HYP (see chat discussion: rank/conditioning != linearizability).

In [ ]:

def effective_rank_and_condition(feats):
    # feats: [n_patches, d]
    feats_c = feats - feats.mean(axis=0, keepdims=True)
    cov = np.cov(feats_c, rowvar=False)
    eigvals = np.linalg.eigvalsh(cov)
    eigvals = np.clip(eigvals, 1e-12, None)
    p = eigvals / eigvals.sum()
    effective_rank = np.exp(-np.sum(p * np.log(p)))  # exponential of entropy
    condition_number = eigvals.max() / eigvals.min()
    return effective_rank, condition_number

diag_rows = []
for which in ['phi_pre', 'phi_post']:
    for cls in CLASS_NAMES:
        feats, _ = load_class_features(cls, which)
        if feats is None:
            continue
        er, cond = effective_rank_and_condition(feats)
        diag_rows.append({'which': which, 'class': cls, 'effective_rank': er,
                           'condition_number': cond, 'ambient_dim': feats.shape[1]})

diag_df = pd.DataFrame(diag_rows)
print(diag_df.to_string(index=False))
diag_df.to_csv('a3_geometry_diagnostics.csv', index=False)


## Section 10 — Summary table + reading against the pre-registered map

Pulls everything into one table and prints the interpretation cell (from Section 0) that
matches the observed pattern — **read this cell's printed output, don't just eyeball the raw
tables**, since the whole point of pre-registering the map was to avoid post-hoc narrative
fitting.

In [ ]:

print('=== A3 SUMMARY ===\n')
print('--- eDMD median residual by class (median across 3 splits) ---')
print(edmd_df.groupby(['which', 'class'])['median_resid'].median().unstack('which'))

print('\n--- eDMD significance (chaotic vs. Burgers), fraction of 3 splits with p<0.05 ---')
if len(sig_df):
    sig_summary = sig_df.groupby(['which', 'class']).apply(
        lambda g: (g['wilcoxon_p'] < 0.05).mean()
    )
    print(sig_summary)

print('\n--- Jacobian Frobenius norm by class (median) ---')
print(jac_df.groupby('class')['fro_norm'].median())

print('\n--- Geometry diagnostics (context only, not standalone evidence) ---')
print(diag_df)

print(\"\"\"
[REMINDER -- read against Section 0's pre-registered interpretation map]
- Chaotic classes (lorenz/rossler/sprottb) showing consistently HIGHER eDMD residual than
  Burgers, replicated across the 3 splits, in BOTH phi_pre and phi_post -> clean mechanistic
  confirmation of A1's pattern.
- Same in phi_pre only, NOT phi_post -> the trained projection is compensating downstream;
  A1's behavioral split needs a different/additional explanation.
- No consistent separation anywhere -> negative result; push to A2a (temporal attention).
- Report at MEDIUM confidence regardless of outcome (see budget/limitations in Section 0) --
  this is a first-pass test, not a definitive mechanism claim.
\"\"\")
